# llama.cpp

Efficient LLM inference in C/C++ - Run LLMs on CPUs and consumer hardware.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

**llama.cpp** is a plain C/C++ implementation of Meta's LLaMA model, optimized for efficient inference on consumer-grade hardware without requiring GPUs.

### What is it?

llama.cpp is a lightweight LLM inference engine that:
- Runs **entirely on CPU** (with optional GPU acceleration)
- Uses **GGUF format** for ultra-efficient quantized models
- Has **zero dependencies** (core C/C++ implementation)
- Supports **Apple Silicon** (Metal GPU acceleration)
- Works on **any platform**: Linux, macOS, Windows, even Android/iOS
- Provides **Python bindings** for easy integration

### Why use it?

Key benefits:
- **No GPU Required**: Run 7B models on laptops with 8GB RAM
- **Extreme Efficiency**: 2-8 bit quantization for minimal memory footprint
- **Broad Compatibility**: Works on nearly any hardware
- **Fast Development**: Active community, frequent updates
- **Privacy First**: Run models 100% locally, no cloud dependencies
- **Low Cost**: Use existing hardware, no GPU investment needed

### When to use it?

llama.cpp is ideal when:
- **No GPU available** or GPU too expensive
- Running on **consumer hardware** (laptops, desktops, edge devices)
- **Privacy is critical** (on-premise, air-gapped environments)
- **Development/prototyping** before investing in GPU infrastructure
- Deploying on **Apple Silicon** (M1/M2/M3 Macs)
- **Cost optimization** - maximize performance per dollar

**Trade-off**: Slower than GPU solutions but democratizes LLM access.

## Key Features

### Core Capabilities of llama.cpp

| Feature | Description | Benefit |
|---------|-------------|----------|
| **CPU-First** | Optimized for CPU inference with SIMD | Run LLMs without GPU |
| **GGUF Format** | Efficient binary format for quantized models | Faster loading, smaller files |
| **K-Quant Support** | 2-8 bit quantization (Q2_K, Q4_K_M, Q8_0, etc.) | 4-16x memory reduction |
| **Apple Metal** | GPU acceleration on M1/M2/M3 chips | 5-10x faster on Mac |
| **CUDA Support** | Optional NVIDIA GPU acceleration | Best of both worlds |
| **Low Memory** | Run 7B models in 4GB RAM with Q4 | Accessible on any laptop |
| **Fast Loading** | mmap support for instant model loading | No startup delay |
| **Context Extension** | Up to 32K context with RoPE scaling | Long document processing |
| **Sampling Methods** | Temperature, top-p, top-k, mirostat, etc. | Flexible generation control |
| **Multi-Platform** | Linux, macOS, Windows, Android, iOS | Deploy anywhere |
| **Server Mode** | OpenAI-compatible API server | Drop-in replacement |
| **Embeddings** | Generate text embeddings | RAG, semantic search |

## Architecture Overview

llama.cpp's architecture emphasizes simplicity and efficiency:

```
┌─────────────────────────────────────────────────────────┐
│                APPLICATION LAYER                        │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐            │
│  │  CLI     │  │  Server  │  │  Python  │            │
│  │  main    │  │  Mode    │  │  Bindings│            │
│  └────┬─────┘  └────┬─────┘  └────┬─────┘            │
└───────┼─────────────┼─────────────┼───────────────────┘
        │             │             │
        └─────────────┴─────────────┘
                      │
┌─────────────────────▼─────────────────────────────────┐
│              LLAMA.CPP CORE (C/C++)                   │
│                                                       │
│  ┌────────────────────────────────────────────────┐  │
│  │         Model Loading & Format               │  │
│  │  • GGUF parser                                │  │
│  │  • mmap for fast loading                      │  │
│  │  • Quantization dequantization                │  │
│  └─────────────────┬──────────────────────────────┘  │
│                    │                                  │
│  ┌─────────────────▼──────────────────────────────┐  │
│  │        Inference Engine                       │  │
│  │  • Transformer layers                         │  │
│  │  • Attention mechanism                        │  │
│  │  • Feed-forward networks                      │  │
│  │  • KV cache management                        │  │
│  └─────────────────┬──────────────────────────────┘  │
│                    │                                  │
│  ┌─────────────────▼──────────────────────────────┐  │
│  │        Sampling & Generation                  │  │
│  │  • Temperature, top-p, top-k                  │  │
│  │  • Mirostat, locally typical sampling        │  │
│  │  • Repetition penalty                         │  │
│  └─────────────────┬──────────────────────────────┘  │
└────────────────────┼──────────────────────────────────┘
                     │
┌────────────────────▼──────────────────────────────────┐
│            BACKEND ACCELERATION                       │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐          │
│  │   CPU    │  │  Metal   │  │   CUDA   │          │
│  │  SIMD    │  │ (Apple)  │  │ (NVIDIA) │          │
│  │AVX2/NEON │  │   GPU    │  │   GPU    │          │
│  └──────────┘  └──────────┘  └──────────┘          │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐          │
│  │  OpenBLAS│  │   ROCm   │  │  Vulkan  │          │
│  │   CLBlast│  │  (AMD)   │  │          │          │
│  └──────────┘  └──────────┘  └──────────┘          │
└───────────────────────────────────────────────────────┘
```

### Key Components

1. **GGUF Format**: Efficient binary format designed for llama.cpp
2. **Quantization**: K-quant methods (Q2_K through Q8_0) for size/quality trade-offs
3. **SIMD Optimization**: Uses AVX2, AVX512, NEON for fast CPU inference
4. **mmap**: Memory-mapped file loading for instant startup
5. **Backends**: Pluggable acceleration (Metal, CUDA, OpenBLAS, etc.)

## Installation

### Prerequisites

- **C++ Compiler**: GCC, Clang, or MSVC
- **Python 3.8+** (for Python bindings)
- **CMake** (for building from source)
- **Optional**: CUDA toolkit (for NVIDIA GPUs)

### Installation via pip (Easiest)

In [ ]:
# Install llama-cpp-python (Python bindings)
# CPU only
# pip install llama-cpp-python

# With CUDA support (NVIDIA GPUs)
# CMAKE_ARGS="-DLLAMA_CUBLAS=on" pip install llama-cpp-python

# With Metal support (Apple Silicon)
# CMAKE_ARGS="-DLLAMA_METAL=on" pip install llama-cpp-python

# With OpenBLAS (CPU acceleration)
# CMAKE_ARGS="-DLLAMA_BLAS=ON -DLLAMA_BLAS_VENDOR=OpenBLAS" pip install llama-cpp-python

import sys
print(f"Python version: {sys.version}")

# Verify installation
# from llama_cpp import Llama
# print("llama-cpp-python installed successfully")

### Building from Source (For latest features)

In [ ]:
# Build llama.cpp from source
build_instructions = '''
# Clone repository
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp

# Build (CPU only)
make

# Or with CUDA
make LLAMA_CUBLAS=1

# Or with Metal (macOS)
make LLAMA_METAL=1

# Test
./main -m models/7B/ggml-model-q4_0.gguf -p "Hello, my name is" -n 128
'''

print("Build instructions:")
print(build_instructions)

## Basic Usage

### Downloading a Model (GGUF format)

In [ ]:
# Download GGUF models from Hugging Face
download_example = '''
# Popular model repositories with GGUF files:
# - TheBloke (most popular, hundreds of models)
# - MaziyarPanahi
# - QuantFactory

# Example: Download Llama-2-7B Q4_K_M quantization
# huggingface-cli download TheBloke/Llama-2-7B-GGUF \\
#   llama-2-7b.Q4_K_M.gguf \\
#   --local-dir ./models \\
#   --local-dir-use-symlinks False

# Or use wget/curl
# wget https://huggingface.co/TheBloke/Llama-2-7B-GGUF/resolve/main/llama-2-7b.Q4_K_M.gguf
'''

print("Download GGUF models:")
print(download_example)
print("\nQuantization guide:")
print("Q2_K: Smallest, ~2GB for 7B (lower quality)")
print("Q4_K_M: Recommended, ~4GB for 7B (good quality)")
print("Q5_K_M: Better, ~5GB for 7B (very good quality)")
print("Q8_0: Best, ~7GB for 7B (near-original quality)")

### Basic Text Generation

In [ ]:
# Basic usage with llama-cpp-python
basic_example = '''
from llama_cpp import Llama

# Load model
llm = Llama(
    model_path="./models/llama-2-7b.Q4_K_M.gguf",
    n_ctx=2048,        # Context window
    n_threads=8,       # Number of CPU threads
    n_gpu_layers=0,    # Number of layers to offload to GPU (0 = CPU only)
)

# Generate text
output = llm(
    "Q: What is the capital of France? A:",
    max_tokens=32,
    temperature=0.7,
    top_p=0.95,
    stop=["Q:", "\n"],
    echo=False
)

print(output["choices"][0]["text"])
'''

print("Basic usage example:")
print(basic_example)

### Chat Completion (OpenAI-Compatible)

In [ ]:
# Chat completion API
chat_example = '''
from llama_cpp import Llama

llm = Llama(
    model_path="./models/llama-2-7b-chat.Q4_K_M.gguf",
    n_ctx=2048,
    n_threads=8,
    chat_format="llama-2"  # Auto-format for chat models
)

# Chat completion
response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain quantum computing in simple terms."}
    ],
    max_tokens=200,
    temperature=0.7
)

print(response["choices"][0]["message"]["content"])
'''

print("Chat completion example:")
print(chat_example)

### Streaming Generation

In [ ]:
# Streaming for real-time output
streaming_example = '''
from llama_cpp import Llama

llm = Llama(model_path="./models/llama-2-7b.Q4_K_M.gguf")

# Stream tokens as they're generated
for chunk in llm(
    "Once upon a time",
    max_tokens=100,
    stream=True
):
    text = chunk["choices"][0]["text"]
    print(text, end="", flush=True)
'''

print("Streaming example:")
print(streaming_example)

## Advanced Features

### 1. GPU Offloading (Hybrid CPU/GPU)

In [ ]:
# Offload layers to GPU for acceleration
gpu_offload = '''
from llama_cpp import Llama

# Offload some layers to GPU, rest on CPU
llm = Llama(
    model_path="./models/llama-2-7b.Q4_K_M.gguf",
    n_gpu_layers=32,  # Offload 32 layers to GPU
    n_ctx=2048,
)

# Benefits:
# - Run larger models with limited VRAM
# - Balance GPU and CPU usage
# - Fallback when model doesn't fit in VRAM

# Guidelines:
# 7B model: ~32 layers total
# 13B model: ~40 layers total
# 70B model: ~80 layers total

# Start with n_gpu_layers=-1 (all layers)
# Reduce if OOM errors occur
'''

print("GPU offloading:")
print(gpu_offload)

### 2. Context Extension (Long Documents)

In [ ]:
# Extended context with RoPE scaling
context_extension = '''
from llama_cpp import Llama

# Extend context beyond training length
llm = Llama(
    model_path="./models/llama-2-7b.Q4_K_M.gguf",
    n_ctx=8192,              # Extended context (original: 4096)
    rope_freq_base=10000,    # RoPE base frequency
    rope_freq_scale=0.5,     # RoPE scaling factor
)

# Use for:
# - Long document summarization
# - Extended conversations
# - Large code analysis

# Trade-off: Slower inference, more memory
'''

print("Context extension:")
print(context_extension)

### 3. Advanced Sampling Methods

In [ ]:
# Fine-grained control over generation
sampling_example = '''
from llama_cpp import Llama

llm = Llama(model_path="./models/llama-2-7b.Q4_K_M.gguf")

output = llm(
    prompt="Write a poem about AI:",
    max_tokens=200,
    
    # Temperature sampling
    temperature=0.8,          # Randomness (0=deterministic, 1=creative)
    
    # Top-p (nucleus) sampling
    top_p=0.95,               # Consider top 95% probable tokens
    
    # Top-k sampling
    top_k=40,                 # Consider top 40 tokens
    
    # Repetition control
    repeat_penalty=1.1,       # Penalize repetition
    
    # Mirostat (perplexity control)
    mirostat_mode=2,          # 0=disabled, 1/2=enabled
    mirostat_tau=5.0,         # Target perplexity
    mirostat_eta=0.1,         # Learning rate
    
    # Locally typical sampling
    typical_p=1.0,            # Typical sampling (1=disabled)
)
'''

print("Advanced sampling:")
print(sampling_example)

### 4. Embeddings Generation

In [ ]:
# Generate embeddings for RAG, search
embeddings_example = '''
from llama_cpp import Llama

# Load model in embedding mode
llm = Llama(
    model_path="./models/llama-2-7b.Q4_K_M.gguf",
    embedding=True  # Enable embedding mode
)

# Generate embeddings
embeddings = llm.embed("This is a test sentence.")
print(f"Embedding shape: {len(embeddings)}")

# Use for:
# - Semantic search
# - RAG (Retrieval Augmented Generation)
# - Document similarity
# - Clustering
'''

print("Embeddings generation:")
print(embeddings_example)

### 5. Server Mode (OpenAI-Compatible API)

In [ ]:
# Run as OpenAI-compatible API server
server_mode = '''
# Start server (command line)
python -m llama_cpp.server \\
  --model ./models/llama-2-7b.Q4_K_M.gguf \\
  --host 0.0.0.0 \\
  --port 8000 \\
  --n_ctx 2048 \\
  --n_threads 8

# Or with GPU offloading
python -m llama_cpp.server \\
  --model ./models/llama-2-7b.Q4_K_M.gguf \\
  --n_gpu_layers 32 \\
  --host 0.0.0.0 \\
  --port 8000

# Client usage (OpenAI SDK)
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"  # llama.cpp doesn't require API key
)

response = client.chat.completions.create(
    model="llama-2-7b",  # Model name doesn't matter
    messages=[{"role": "user", "content": "Hello!"}]
)

print(response.choices[0].message.content)
'''

print("Server mode (OpenAI-compatible):")
print(server_mode)

## Use Cases

### Real-World Applications

#### Use Case 1: Local Development Environment

In [ ]:
# Run LLMs locally for development
dev_setup = '''
# Scenario: Developer building AI features without cloud dependency

# Setup
from llama_cpp import Llama

llm = Llama(
    model_path="./models/codellama-7b.Q4_K_M.gguf",  # Code model
    n_ctx=4096,
    n_threads=8,
)

# Code completion
def get_code_completion(prompt):
    return llm(
        prompt,
        max_tokens=200,
        temperature=0.2,  # Low temp for deterministic code
        stop=["\n\n", "###"]
    )

# Benefits:
# - No API costs during development
# - Works offline
# - Fast iteration
# - Full privacy (code stays local)
'''

print("Local development setup:")
print(dev_setup)

#### Use Case 2: Privacy-Sensitive Applications

In [ ]:
# On-premise LLM for sensitive data
privacy_setup = '''
# Scenario: Healthcare/legal/financial app with strict privacy requirements

# Deploy llama.cpp on-premise
# - No data leaves the organization
# - Air-gapped deployment possible
# - HIPAA/GDPR compliant

# Example: Medical document analysis
from llama_cpp import Llama

llm = Llama(
    model_path="/secure/models/meditron-7b.Q4_K_M.gguf",
    n_ctx=2048,
)

def analyze_medical_record(record_text):
    prompt = f"""Extract key medical information from:
{record_text}

Key findings:"""
    return llm(prompt, max_tokens=300)

# All processing happens locally
# No API calls, no external dependencies
'''

print("Privacy-sensitive setup:")
print(privacy_setup)

#### Use Case 3: Edge Deployment (Raspberry Pi, Mobile)

In [ ]:
# Run on resource-constrained devices
edge_deployment = '''
# Scenario: IoT device with local AI capabilities

# Use small quantized model
from llama_cpp import Llama

# Tiny model for edge devices
llm = Llama(
    model_path="./models/tinyllama-1.1b.Q4_K_M.gguf",  # 1B model
    n_ctx=512,          # Smaller context
    n_threads=4,        # Limited threads
    use_mmap=True,      # Memory-map for efficiency
    use_mlock=False,    # Don't lock memory
)

# Memory footprint:
# TinyLlama 1.1B Q4: ~700MB RAM
# Llama-2 7B Q2: ~2.5GB RAM
# Llama-2 7B Q4: ~4GB RAM

# Deployment targets:
# - Raspberry Pi 4/5
# - Android phones
# - Edge servers
# - IoT gateways
'''

print("Edge deployment:")
print(edge_deployment)

## Best Practices

### Recommended Practices for llama.cpp

#### 1. Choosing Quantization Level

```python
# Quality vs Size trade-off guide:

# Q2_K: Smallest (2-bit)
# - Use when: Extreme memory constraints
# - Quality: Noticeable degradation
# - Size: ~2GB for 7B model

# Q3_K_M: Small (3-bit)
# - Use when: Very limited RAM
# - Quality: Some degradation
# - Size: ~3GB for 7B model

# Q4_K_M: Recommended (4-bit)
# - Use when: Balanced needs
# - Quality: Good, minimal degradation
# - Size: ~4GB for 7B model
# - BEST CHOICE for most use cases

# Q5_K_M: High Quality (5-bit)
# - Use when: Quality matters more than size
# - Quality: Very good
# - Size: ~5GB for 7B model

# Q8_0: Maximum Quality (8-bit)
# - Use when: Need near-original quality
# - Quality: Nearly identical to FP16
# - Size: ~7GB for 7B model
```

#### 2. CPU Optimization

- **Use all available threads**: `n_threads = os.cpu_count()` for maximum CPU utilization
- **Enable BLAS**: Build with OpenBLAS or Apple Accelerate for 2-3x speedup
- **Use mmap**: `use_mmap=True` for fast loading and memory efficiency
- **Batch processing**: Process multiple prompts together when possible

#### 3. Context Management

- Start with `n_ctx=2048` (standard)
- Only increase if you need longer context (uses more RAM)
- Monitor memory usage when extending context
- Use `rope_freq_scale` for context beyond training length

#### 4. Memory Management

- Reserve ~1-2GB system RAM beyond model size
- Use `use_mlock=False` unless you need guaranteed memory
- Close model with `del llm` when done to free memory
- Monitor swap usage - avoid swapping at all costs

#### 5. Deployment

- Use server mode for multi-user scenarios
- Pin model files to fast SSD for quick loading
- Set reasonable `max_tokens` limits
- Implement request queuing for concurrent users

## Common Pitfalls

### What to Avoid

#### 1. Using Wrong Quantization Format

**Problem**: Trying to load old GGML files instead of GGUF

**Symptom**: `Invalid GGUF file` or `Failed to load model`

**Solution**: llama.cpp moved to GGUF format in 2023. Always download `.gguf` files, not `.ggml` or `.bin`

#### 2. Insufficient RAM

**Problem**: Model + context exceeds available RAM

**Symptom**: System swap thrashing, extremely slow inference

**Solution**:
```python
# Use smaller quantization
# Q4_K_M instead of Q8_0

# Or reduce context
n_ctx = 512  # instead of 2048

# Or use smaller model
# 3B instead of 7B
```

#### 3. Not Using Threads Effectively

**Problem**: Using only 1-2 threads on multi-core CPU

**Impact**: 4-8x slower than optimal

**Solution**:
```python
import os
llm = Llama(
    model_path="model.gguf",
    n_threads=os.cpu_count()  # Use all cores
)
```

#### 4. Ignoring Context Overflow

**Problem**: Prompt + output exceeds `n_ctx`

**Symptom**: Truncated context, degraded quality

**Solution**: Monitor context usage, implement truncation strategy

#### 5. Wrong Chat Format

**Problem**: Not using correct prompt format for chat models

**Impact**: Poor quality responses

**Solution**: Use `chat_format` parameter or manually format prompts:
```python
llm = Llama(
    model_path="llama-2-7b-chat.gguf",
    chat_format="llama-2"  # or "chatml", "alpaca", etc.
)
```

## Performance Optimization

### Achieving Maximum Speed

**CPU Optimization Hierarchy** (from most to least impact):

1. **Use smaller quantization** (Q4 vs Q8): 2x faster, minimal quality loss
2. **Enable BLAS libraries**: 2-3x speedup
3. **Use all CPU threads**: Linear scaling with cores
4. **GPU offloading** (if available): 5-10x faster than CPU
5. **Reduce context window**: Lower memory bandwidth

**Benchmark Your Setup**:

In [ ]:
# Benchmark llama.cpp performance
benchmark_code = '''
import time
from llama_cpp import Llama

llm = Llama(model_path="model.gguf", n_threads=8)

# Benchmark prompt processing
prompt = "Hello " * 100  # ~100 tokens
start = time.time()
llm(prompt, max_tokens=1, echo=False)
prompt_time = time.time() - start
print(f"Prompt processing: {len(prompt.split())/prompt_time:.0f} tokens/sec")

# Benchmark generation
start = time.time()
output = llm("Test", max_tokens=100, echo=False)
gen_time = time.time() - start
print(f"Generation: {100/gen_time:.1f} tokens/sec")

# Typical speeds (CPU, Llama-2 7B Q4):
# M1 Max: ~20-30 tokens/sec
# Intel i9-13900K: ~15-25 tokens/sec
# AMD Ryzen 9 7950X: ~15-25 tokens/sec
# Raspberry Pi 5: ~1-2 tokens/sec
'''

print("Benchmark code:")
print(benchmark_code)

## Production Deployment

### Docker Deployment

In [ ]:
# Production Dockerfile
dockerfile = '''
FROM python:3.11-slim

# Install build dependencies
RUN apt-get update && apt-get install -y \\
    build-essential \\
    cmake \\
    libopenblas-dev \\
    && rm -rf /var/lib/apt/lists/*

# Install llama-cpp-python with OpenBLAS
RUN CMAKE_ARGS="-DLLAMA_BLAS=ON -DLLAMA_BLAS_VENDOR=OpenBLAS" \\
    pip install llama-cpp-python[server]

# Copy model
COPY models/ /models/

# Health check
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
  CMD curl -f http://localhost:8000/health || exit 1

# Run server
CMD ["python", "-m", "llama_cpp.server", \\
     "--model", "/models/llama-2-7b.Q4_K_M.gguf", \\
     "--host", "0.0.0.0", \\
     "--port", "8000", \\
     "--n_ctx", "2048", \\
     "--n_threads", "8"]
'''

print("Production Dockerfile:")
print(dockerfile)

## Comparison with Alternatives

### How llama.cpp Compares

| Feature | llama.cpp | Ollama | vLLM | TensorRT-LLM |
|---------|-----------|--------|------|---------------|
| **CPU Support** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐ | ⭐ |
| **GPU Support** | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Ease of Use** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ |
| **Memory Efficiency** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Model Support** | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ |
| **Speed (CPU)** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐ | ⭐ |
| **Speed (GPU)** | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Deployment** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ |

### When to Choose llama.cpp

**Choose llama.cpp when:**
- ✅ **No GPU available** or GPU too expensive
- ✅ Running on **consumer hardware** (laptops, desktops)
- ✅ **Privacy is critical** (fully local deployment)
- ✅ **Apple Silicon** (M1/M2/M3 optimization)
- ✅ **Edge deployment** (low-power devices)
- ✅ Need **maximum flexibility** in deployment

**Choose alternatives when:**
- ❌ Have powerful GPUs (use vLLM or TensorRT-LLM)
- ❌ Need absolute maximum throughput (GPU solutions)
- ❌ Want simplest setup (use Ollama)
- ❌ Production serving at scale (consider GPU solutions)

## Resources

### Official Documentation

- **GitHub**: https://github.com/ggerganov/llama.cpp
- **Python Bindings**: https://github.com/abetlen/llama-cpp-python
- **GGUF Format Spec**: https://github.com/ggerganov/ggml/blob/master/docs/gguf.md
- **Model Downloads**: https://huggingface.co/TheBloke (most comprehensive)

### Community Resources

- **Reddit**: r/LocalLLaMA
- **Discord**: llama.cpp community server
- **Discussions**: https://github.com/ggerganov/llama.cpp/discussions

### Model Repositories

- **TheBloke**: https://huggingface.co/TheBloke (hundreds of GGUF models)
- **MaziyarPanahi**: https://huggingface.co/MaziyarPanahi
- **QuantFactory**: https://huggingface.co/QuantFactory

### Tools and Utilities

- **Model Converters**: llama.cpp/convert.py
- **Quantization Tools**: llama.cpp/quantize
- **Perplexity Testing**: llama.cpp/perplexity

### Related Projects

- **Ollama**: User-friendly wrapper around llama.cpp
- **LM Studio**: GUI for llama.cpp
- **koboldcpp**: llama.cpp with story generation features
- **text-generation-webui**: Web UI for llama.cpp